In [ ]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv

# === Load Tokens from .env ===
load_dotenv("D:/Android_Mobile_App/AndroidProject_dataset/All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "manifest-checker"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Rotated to token #{token_index + 1}")

# === GitHub request with token rotation ===
def makeRequest(url):
    tries_flag = 0
    while True:
        response = requests.get(url, headers=get_headers())
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 403 and "rate limit" in response.json().get("message", "").lower():
            rotate_token()
            tries_flag += 1
            if tries_flag > len(tokens):
                reset_timestamp = int(response.headers.get('X-RateLimit-Reset', time.time() + 60))
                wait_time = reset_timestamp - int(time.time())
                print(f"Rate limit exceeded. Waiting {wait_time} seconds...")
                time.sleep(wait_time + 1)
                tries_flag = 0
            else:
                time.sleep(1)
        elif response.status_code in [404, 422]:
            return {"total_count": 0}
        else:
            print(f"Request failed with status code {response.status_code}. Retrying...")
            time.sleep(1)

# === Owner/repo extractor ===
def extract_owner_repo(url):
    try:
        parts = url.strip().split("github.com/")[-1].replace(".git", "").split("/")
        return parts[0], parts[1]
    except:
        return None, None

# === Load CSV ===
csv_path = "D:/Android_Mobile_App/AndroidProject_dataset/Repo_List_OR.csv"
df = pd.read_csv(csv_path)

# === Add Columns ===
df["has_manifest"] = "no"
df["has_activity"] = "no"

# === Process each repository ===
search_base_url = "https://api.github.com/search/code?q=activity+filename:AndroidManifest.xml+repo:"
for i, row in df.iterrows():
    owner, repo = extract_owner_repo(row["html_url"])
    if not owner or not repo:
        continue
    response = makeRequest(search_base_url + f"{owner}/{repo}")
    df.at[i, "has_manifest"] = "yes" if response.get("total_count", 0) > 0 else "no"
    df.at[i, "has_activity"] = "yes" if response.get("total_count", 0) > 0 else "no"

# === Save Output ===
output_path = "D:/Android_Mobile_App/AndroidProject_dataset/Repo_List_OR_with_manifest.csv"
df.to_csv(output_path, index=False)
print(f"✅ File saved to: {output_path}")
